<a href="https://colab.research.google.com/github/ABI0508/covid19-chest-xray-resnet50/blob/main/residual_rebels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
tawsifurrahman_covid19_radiography_database_path = kagglehub.dataset_download('tawsifurrahman/covid19-radiography-database')
abikullachi_validation_path = kagglehub.dataset_download('abikullachi/validation')
abikullachi_resnet50_keras_default_1_path = kagglehub.model_download('abikullachi/resnet50/Keras/default/1')

print('Data source import complete.')


Data Loading & Masked Segmentation

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.model_selection import train_test_split

IMG_SIZE = 224
base_path = '/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset'
categories = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']

def load_and_preprocess_segmented(image_path, mask_path):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    mask = cv2.imread(mask_path, 0)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))


    segmented_img = cv2.bitwise_and(img, img, mask=mask)


    return preprocess_input(segmented_img.astype('float32'))

X, y = [], []
print("🚀 Phase 1: Loading & Segmenting Data (1000/category)...")
for idx, cat in enumerate(categories):
    img_dir, mask_dir = os.path.join(base_path, cat, 'images'), os.path.join(base_path, cat, 'masks')
    files = sorted(os.listdir(img_dir))[:1000]
    print(f"✅ Processing {cat}...")
    for f in files:
        img_p, mask_p = os.path.join(img_dir, f), os.path.join(mask_dir, f)
        if os.path.exists(mask_p):
            X.append(load_and_preprocess_segmented(img_p, mask_p))
            y.append(idx)

X = np.array(X)
y = tf.keras.utils.to_categorical(np.array(y), 4)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
print("✓ Data Loaded Successfully!")

Building Optimized ResNet-50


In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

# Unfreezing last 100 layers for specialized learning
for layer in base_model.layers[:-100]:
    layer.trainable = False
for layer in base_model.layers[-100:]:
    layer.trainable = True

x = GlobalAveragePooling2D()(base_model.output)
x = BatchNormalization()(x)
x = Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
x = Dropout(0.5)(x)
predictions = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
print("✅ High-Performance Model Built!")

Training with Data Augmentation

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='constant',
    cval=0
)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-7, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1)

print("\n🔥 Training Started...")
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    validation_data=(X_val, y_val),
    epochs=40,
    callbacks=[reduce_lr, early_stop]
)

Performance Analytics

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

y_pred = model.predict(X_val)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_val, axis=1)

# Confusion Matrix Visual
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_true, y_pred_classes), annot=True, fmt='d', cmap='Blues', xticklabels=categories, yticklabels=categories)
plt.title('Final Confusion Matrix')
plt.show()

# Text Report
print(classification_report(y_true, y_pred_classes, target_names=categories))

Curves



In [ ]:
import matplotlib.pyplot as plt

# Plotting Accuracy and Loss Curves
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(15, 5))

# 1. Accuracy Curve
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy', color='#2ecc71', linewidth=2)
plt.plot(epochs_range, val_acc, label='Validation Accuracy', color='#e67e22', linewidth=2)
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)

# 2. Loss Curve
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss', color='#e74c3c', linewidth=2)
plt.plot(epochs_range, val_loss, label='Validation Loss', color='#3498db', linewidth=2)
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)

plt.show()

Master Download Bundle

In [ ]:
import zipfile
import os
import numpy as np
import cv2
from IPython.display import FileLink

# 1. Save the Model (H5 Format)
model.save('msc_best_covid_model.h5')

np.save('X_val.npy', X_val)
np.save('y_val.npy', y_val)

# 3. Create a folder for Visual Sample Images (For your Project Report)
os.makedirs('segmented_samples', exist_ok=True)
for i in range(15):  # Saving 15 samples
    # De-normalizing for PNG saving
    sample_img = (X_val[i] - X_val[i].min()) / (X_val[i].max() - X_val[i].min() + 1e-10)
    sample_img = (sample_img * 255).astype('uint8')
    # Save as BGR for OpenCV
    cv2.imwrite(f'segmented_samples/sample_{i}.png', cv2.cvtColor(sample_img, cv2.COLOR_RGB2BGR))

# 4. ZIP everything into one Master File
bundle_name = 'MSc_Final_Project_Full_Bundle.zip'
with zipfile.ZipFile(bundle_name, 'w') as zipf:
    # Adding model and arrays
    zipf.write('msc_best_covid_model.h5')
    zipf.write('X_val.npy')
    zipf.write('y_val.npy')

    # Adding the segmented samples folder
    for root, dirs, files in os.walk('segmented_samples'):
        for file in files:
            zipf.write(os.path.join(root, file))

print("✅ SUCCESS: All project files are compressed!")
print("👇 Click the link below to download the ZIP to your laptop:")
display(FileLink(bundle_name))

t-SNE Visualization (Global Feature Space)

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.manifold import TSNE
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

try:
    # 1. Variable Load
    if 'model' not in locals():
        model = tf.keras.models.load_model('/kaggle/input/models/abikullachi/resnet50/keras/default/1/msc_best_covid_model.h5')
    if 'X_val' not in locals():
        X_val = np.load('/kaggle/input/datasets/abikullachi/validation/X_val.npy')
    if 'y_val' not in locals():
        y_val = np.load('/kaggle/input/datasets/abikullachi/validation/y_val.npy')

    categories = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']

    # 2. Extract Features using the CORRECT layer name: 'global_average_pooling2d'
    print("Extracting features from 'global_average_pooling2d' layer...")
    feature_extractor = tf.keras.models.Model(
        inputs=model.input,
        outputs=model.get_layer('global_average_pooling2d').output
    )
    features = feature_extractor.predict(X_val, verbose=1)

    # 3. Run t-SNE
    print("Running t-SNE algorithm...")
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    tsne_results = tsne.fit_transform(features)

    # 4. Visualization
    df_tsne = pd.DataFrame(tsne_results, columns=['Dim 1', 'Dim 2'])
    df_tsne['Label'] = [categories[np.argmax(y)] for y in y_val]

    plt.figure(figsize=(10, 7))
    sns.scatterplot(x="Dim 1", y="Dim 2", hue="Label", data=df_tsne, palette="bright", alpha=0.7)
    plt.title("MSc Project: t-SNE Cluster Mapping (Global Average Pooling Features)")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

except Exception as e:
    print(f"Error logic check: {e}")

XAI (Dual-Mode Interactive UI) for block 3 Final Layer

In [ ]:
import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from tensorflow.keras.applications.resnet50 import preprocess_input

# --- STEP 1: LOAD ASSETS ---

MODEL_PATH = '/kaggle/input/models/abikullachi/resnet50/keras/default/1/msc_best_covid_model.h5'
X_VAL_PATH = '/kaggle/input/datasets/abikullachi/validation/X_val.npy'
Y_VAL_PATH = '/kaggle/input/datasets/abikullachi/validation/y_val.npy'

try:
    model = tf.keras.models.load_model(MODEL_PATH)
    X_val = np.load(X_VAL_PATH)
    y_val = np.load(Y_VAL_PATH)
    categories = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
    print("✅ Model, X_val, and y_val loaded successfully!")
except Exception as e:
    print(f"❌ Error loading files: {e}")
    print("Check if the file paths are correct in the '/kaggle/input/' directory.")

# --- STEP 2: GRAD-CAM LOGIC ---
def get_gradcam(img_array, model):
    # ResNet50 specific layer
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer('conv5_block3_out').output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        class_channel = preds[:, tf.argmax(preds[0])]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()

# --- STEP 3: ANALYSIS & PLOTTING ---
def run_xai_analysis(img_input, title_label="Sample"):
    preds = model.predict(img_input, verbose=0)
    p_idx = np.argmax(preds[0])
    conf = preds[0][p_idx] * 100

    heatmap = get_gradcam(img_input, model)
    heatmap_res = cv2.resize(heatmap, (224, 224))
    heatmap_rgb = cv2.applyColorMap(np.uint8(255 * heatmap_res), cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap_rgb, cv2.COLOR_BGR2RGB)

    # Normalizing for display
    disp_img = img_input[0].copy()
    disp_img = (disp_img - disp_img.min()) / (disp_img.max() - disp_img.min() + 1e-10)
    superimposed_img = cv2.addWeighted((disp_img * 255).astype('uint8'), 0.6, heatmap_rgb, 0.4, 0)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(disp_img)
    plt.title(f"Input: {title_label}")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(superimposed_img)
    plt.title(f"XAI Result: {categories[p_idx]} ({conf:.1f}%)")
    plt.axis('off')
    plt.show()

# --- STEP 4: UI COMPONENTS ---
out = widgets.Output()
idx_input = widgets.IntText(value=0, description='Index:', layout={'width': '150px'})
btn_predict = widgets.Button(description='Predict Dataset Index', button_style='primary')
file_upload = widgets.FileUpload(accept='.png, .jpg, .jpeg', multiple=False, description='Upload X-Ray')

def on_predict_clicked(b):
    with out:
        clear_output()
        idx = idx_input.value
        if idx < len(X_val):
            actual = categories[np.argmax(y_val[idx])]
            run_xai_analysis(X_val[idx:idx+1], title_label=f"Dataset Index {idx} (Actual: {actual})")
        else:
            print(f"Index {idx} out of range!")

def on_upload_change(change):
    with out:
        clear_output()
        if not file_upload.value: return
        uploaded_file = file_upload.value[0]
        nparr = np.frombuffer(uploaded_file['content'], np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img, (224, 224))
        img_pre = preprocess_input(img_resized.astype('float32'))
        run_xai_analysis(np.expand_dims(img_pre, axis=0), title_label="Manual Upload")

btn_predict.on_click(on_predict_clicked)
file_upload.observe(on_upload_change, names='value')

# --- DISPLAY UI ---
print("\n🛡️ MSC PROJECT: XAI DIAGNOSIS INTERFACE")
display(widgets.VBox([
    widgets.HTML("<b>1. Enter Dataset Index (0 to 599):</b>"),
    widgets.HBox([idx_input, btn_predict]),
    widgets.HTML("<br><b>2. Or Upload a New X-Ray Image:</b>"),
    file_upload,
    out
]))

\Block 2-Intermediate Features![image.png](attachment:d85e2263-a50c-4646-929c-47007e5719a0.png)  ![image.png](attachment:c8fa73b5-11d2-4626-9cc4-44f8bf0f274c.png)

In [ ]:
import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from tensorflow.keras.applications.resnet50 import preprocess_input

# --- STEP 1: LOAD ASSETS ---

MODEL_PATH = '/kaggle/input/models/abikullachi/resnet50/keras/default/1/msc_best_covid_model.h5'
X_VAL_PATH = '/kaggle/input/datasets/abikullachi/validation/X_val.npy'
Y_VAL_PATH = '/kaggle/input/datasets/abikullachi/validation/y_val.npy'

try:
    model = tf.keras.models.load_model(MODEL_PATH)
    X_val = np.load(X_VAL_PATH)
    y_val = np.load(Y_VAL_PATH)
    categories = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
    print("✅ Model loaded with SHARP XAI (Block 2) logic!")
except Exception as e:
    print(f"❌ Error loading files: {e}")

# --- STEP 2: GRAD-CAM LOGIC (SHARP VERSION) ---
def get_gradcam(img_array, model):

    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer('conv5_block2_out').output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        class_channel = preds[:, tf.argmax(preds[0])]


    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)


    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()

# --- STEP 3: ANALYSIS & PLOTTING ---
def run_xai_analysis(img_input, title_label="Sample"):
    preds = model.predict(img_input, verbose=0)
    p_idx = np.argmax(preds[0])
    conf = preds[0][p_idx] * 100

    heatmap = get_gradcam(img_input, model)
    heatmap_res = cv2.resize(heatmap, (224, 224))
    heatmap_rgb = cv2.applyColorMap(np.uint8(255 * heatmap_res), cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap_rgb, cv2.COLOR_BGR2RGB)

    # Display images normalization
    disp_img = img_input[0].copy()
    disp_img = (disp_img - disp_img.min()) / (disp_img.max() - disp_img.min() + 1e-10)
    superimposed_img = cv2.addWeighted((disp_img * 255).astype('uint8'), 0.6, heatmap_rgb, 0.4, 0)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(disp_img)
    plt.title(f"Input: {title_label}")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(superimposed_img)
    plt.title(f"XAI (Block 2): {categories[p_idx]} ({conf:.1f}%)")
    plt.axis('off')
    plt.show()

# --- STEP 4: UI COMPONENTS ---
out = widgets.Output()
idx_input = widgets.IntText(value=0, description='Index:', layout={'width': '150px'})
btn_predict = widgets.Button(description='Predict Dataset Index', button_style='primary')
file_upload = widgets.FileUpload(accept='.png, .jpg, .jpeg', multiple=False, description='Upload X-Ray')

def on_predict_clicked(b):
    with out:
        clear_output()
        idx = idx_input.value
        if idx < len(X_val):
            actual = categories[np.argmax(y_val[idx])]
            run_xai_analysis(X_val[idx:idx+1], title_label=f"Dataset Index {idx} (Actual: {actual})")
        else:
            print(f"Index {idx} out of range!")

def on_upload_change(change):
    with out:
        clear_output()
        if not file_upload.value: return
        uploaded_file = file_upload.value[0]
        nparr = np.frombuffer(uploaded_file['content'], np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img, (224, 224))
        img_pre = preprocess_input(img_resized.astype('float32'))
        run_xai_analysis(np.expand_dims(img_pre, axis=0), title_label="Manual Upload")

btn_predict.on_click(on_predict_clicked)
file_upload.observe(on_upload_change, names='value')

# --- DISPLAY UI ---
print("\n🛡️ MSC PROJECT: SHARP XAI INTERFACE (BLOCK 2)")
display(widgets.VBox([
    widgets.HTML("<b>1. Enter Dataset Index (0 to 599):</b>"),
    widgets.HBox([idx_input, btn_predict]),
    widgets.HTML("<br><b>2. Or Upload a New X-Ray Image:</b>"),
    file_upload,
    out
]))